# JAX basics

A short tour of the JAX features used in this repository, before `version1.py`: device arrays, `jit`, `grad`, `vmap`, higher-order derivatives, jaxprs, and two gotchas. Timings depend on your hardware.

In [ ]:
import numpy as np
import jax.numpy as jnp
from matplotlib import pyplot as plt
from jax import jit, vmap, grad, random, make_jaxpr, device_put

## Arrays and devices
`jax.random` needs an explicit PRNG key (no global seed). `device_put` moves a numpy array to the default device (GPU/TPU if available) once.

In [ ]:
seed = 0
size = 3000
key = random.PRNGKey(seed)

x_jnp = random.normal(key, (size, size), dtype=jnp.float32)
x_np = np.random.normal(size=(size, size)).astype(np.float32)
x_np_device = device_put(x_np)

Matrix multiplication: JAX vs numpy. JAX calls are asynchronous, so `.block_until_ready()` is needed for honest timings. Passing a numpy array to a JAX function copies it to the device on every call; pre-placing it with `device_put` avoids that.

In [ ]:
%timeit jnp.dot(x_jnp, x_jnp.T).block_until_ready()
%timeit np.dot(x_np, x_np.T)
%timeit jnp.dot(x_np, x_np.T).block_until_ready()
%timeit jnp.dot(x_np_device, x_np_device.T).block_until_ready()

## `jit`: compiling a function with XLA
The SELU activation, written with `jnp.where` (no Python `if` on array values, which `jit` could not trace).

In [ ]:
def selu(x, alpha=1.67, scale=1.05):
    return scale * jnp.where(x > 0, x, alpha * jnp.exp(x) - alpha)

jit_selu = jit(selu)

In [ ]:
def visualize(func):
    x = jnp.linspace(-10, 10, 1000)
    y = func(x)
    plt.plot(x, y)
    plt.title(func.__name__)
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.grid()
    plt.show()

visualize(selu)

The jitted version fuses the element-wise operations into one kernel.

In [ ]:
data = random.normal(key, (1_000_000,))

print('non jit version')
%timeit selu(data).block_until_ready()
print('jit version')
%timeit jit_selu(data).block_until_ready()

## `grad` and `vmap`
`grad` returns the derivative of a scalar function; `vmap` maps it over a batch of inputs. The derivative of the logistic function is $\sigma(x)(1-\sigma(x))$.

In [ ]:
def logistic(x):
    return 1 / (1 + jnp.exp(-x))

visualize(logistic)

grad_logistic = grad(logistic)

visualize(vmap(grad_logistic))

## Jacobians and Hessians
`jacfwd` (forward mode) and `jacrev` (reverse mode) compose, so a Hessian is the forward-mode Jacobian of the reverse-mode gradient.

In [ ]:
from jax import jacfwd, jacrev
f = lambda x, y: x ** 2 + y ** 2

def hessian(f):
    return jit(jacfwd(jacrev(f, argnums=(0, 1)), argnums=(0, 1)))

print(f'Jacobian = {jacfwd(f, argnums=(0, 1))(1.0, 1.0)}')
print(f'Hessian = {hessian(f)(1.0, 1.0)}')

## Under the hood: the jaxpr
`make_jaxpr` shows the traced program that `jit` hands to XLA.

In [ ]:
print(make_jaxpr(selu)(5))

## Gotchas
Division by zero gives `nan` instead of raising an error.

In [ ]:
jnp.divide(0, 0)

Out-of-bounds indexing does not raise either. JAX cannot raise errors from compiled device code, so the index is clamped (`arr[3]` returns the last element on CPU) or, on some backends, an arbitrary value comes back. Check your indices yourself.

In [ ]:
arr = jnp.array([1, 2, 3])
print(arr)
print(arr[2])
print(arr[3])